# 📈 Day Trader Dashboard

Führe die Zellen **nacheinander** aus (▶ Button links neben jeder Zelle).

Am Ende bekommst du einen Link – den im Browser öffnen.

In [ ]:
# Zelle 1: Pakete installieren (dauert ~1-2 Minuten)
!pip install -q streamlit yfinance ta plotly pandas numpy streamlit-autorefresh pyngrok

In [ ]:
# Zelle 2: Code herunterladen
import os
if os.path.exists('sonstiges'):
    !cd sonstiges && git pull -q
    print('✅ Code aktualisiert')
else:
    !git clone -q https://github.com/reulzinger/sonstiges
    print('✅ Code heruntergeladen')

In [ ]:
# Zelle 3: ngrok-Token eintragen
# 1. Kostenlos registrieren: https://dashboard.ngrok.com/signup
# 2. Token kopieren von: https://dashboard.ngrok.com/get-started/your-authtoken
# 3. Zwischen die Anführungszeichen einfügen

NGROK_TOKEN = "HIER_DEINEN_TOKEN_EINFÜGEN"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ Token gesetzt')

In [ ]:
# Zelle 4: Streamlit testen (zeigt Fehler falls vorhanden)
import subprocess
result = subprocess.run(
    ["python", "-c", "import sys; sys.path.insert(0,'sonstiges/day_trader'); import app"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('❌ Fehler beim Import:')
    print(result.stderr)
else:
    print('✅ Import-Test bestanden')

In [ ]:
# Zelle 5: Dashboard starten
import subprocess, threading, time, socket

streamlit_log = []

def run_streamlit():
    proc = subprocess.Popen(
        [
            "streamlit", "run",
            "sonstiges/day_trader/app.py",
            "--server.port=8501",
            "--server.headless=true",
            "--server.enableCORS=false",
            "--server.enableXsrfProtection=false",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        streamlit_log.append(line)

def warte_bis_bereit(port=8501, timeout=60):
    print('⏳ Warte bis Streamlit startet...', end='', flush=True)
    start = time.time()
    while time.time() - start < timeout:
        try:
            with socket.create_connection(('localhost', port), timeout=1):
                print(' bereit!', flush=True)
                return True
        except (ConnectionRefusedError, OSError):
            print('.', end='', flush=True)
            time.sleep(1)
    print(' Timeout!', flush=True)
    return False

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

if warte_bis_bereit():
    public_url = ngrok.connect(8501)
    print()
    print('✅ Dashboard läuft!')
    print()
    print(f'👉 Link: {public_url}')
    print()
    print('Den Link im Browser öffnen. Zelle läuft im Hintergrund weiter.')
    while True:
        time.sleep(60)
else:
    print()
    print('❌ Streamlit hat nicht gestartet. Fehlermeldung:')
    print(''.join(streamlit_log[-30:]) or '(kein Output)')